In [ ]:
import math
import ast
import ipynbname
import pandas as pd
import numpy as np
import optuna
from Testing.DRTLO_QN_R1 import *
from Functions.AutoCloud_V2 import *
from Functions.DataCloud_V2 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *

FileName = ipynbname.name()
out_path = f'Optimization\\eDRTLO_QN\\multi\\Optimization.csv'
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = HI['PC1'].values

In [ ]:
params =  {'nI': 3, 'nR': 3, 'nO': 3, 'N1': 0.1,
            'τ': 23, 'mode': 1, 'act': 0}

nI,nR,nO,N1,τ,mode,act= list(params.values())
X, Y, Z = PrepareData(RS, HI, nX=nI, nY=nO, mY=0, nZ=1, mode='past')

teda=AutoCloud(m=2.5,nI=nI,nR=[nR],nO=nO,ηS=[N1],mode=mode,
               tau=τ,rho=0.01,eol=0.3,ref=len(sig)-nI,wtaG=True,wtaP=True) 
for j,_ in enumerate(X[:]):
    teda.run(X[j])
    teda.RUL_Prediction(Y[j],mode='interval',lim=len(sig)-nI+1,show=False)
    teda.Adapt(Y[j],Z[j])

#teda.RUL_Granularity()
#print(teda.wape_HI,teda.wape_RUL)
#PlotDSI_3D_PLT(teda)
#teda.c = np.append(teda.c,teda.gm)   
#PlotGranulesSpace(teda,w=700,h=700)

In [ ]:
print(xxx)

In [14]:
def Optimize(
        FileName=None,OptDim=None,OptSampler=None,OptSeed=None,OptPrune=False,
        n_study=None,timeout=None,n_trials=None,
        patience=None,
        mS =None,nIS=None,nLS=None,nRS=None,nOS =None,mOS=None,
        NS=None,τS =None,mdS=None,actS=None):

    '''
        OptSampler modes available: None, Auto, GP, NSGAII, Random, TPE
    
        None: sampler default, suport multivariate optimization

        GP: suport multivariate optimization

        Auto: suport multivariate optimization
        
        NSGAII: poorly suport multivariate optimization
        
        Random: poorly suport multivariate optimization

        TPE: suport multivariate optimization
    '''

    if OptDim == 1 or OptDim is None: SingleObj = True
    elif OptDim > 1: SingleObj = False    
    if n_study is None: n_study = 1
    if timeout is None: timeout = 60
    if n_trials is None and patience is None: 
        n_trials = 25
        patience = 25
    elif n_trials < 100 and patience is None:
        patience = int(0.25*n_trials)
    elif n_trials >= 1e2 and n_trials < 1e3 and patience is None:
        patience = int(0.20*n_trials)
    elif n_trials >= 1e3 and n_trials < 5e3 and patience is None:
        patience = int(0.150*n_trials)
    elif n_trials >= 5e3 and n_trials < 1e4 and patience is None:
        patience = int(0.125*n_trials)    
    elif n_trials >= 1e4 and patience is None:
        patience = int(0.03*n_trials)   

    if mS  is None: mS  = [1.75,4.25]
    if nIS is None: nIS = [2,40]
    if nLS is None: nLS = [1,5]
    if nRS is None: nRS = [1,60]
    if nOS is None: nOS = [1,40]
    if mOS is None: mOS = [0,40]
    if NS is None: NS = [1,9]
    if τS  is None: τS  = [1,25]
    if mdS is None: mdS = [0,1]
    if actS is None: actS = [0,1,2]

    if SingleObj: names = ['MAPE_RUL*MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N','TAU','past/ahead','activation']
    else: names = ['MAPE_RUL','MAPE_HI', 'm', 'nI', 'nR', 'nO', 'mO', 'N', 'TAU','past/ahead','activation']
    study_dir, out_path = df_ParamsTable(names,FileName)

    for i in range(n_study):
        print('iteration:',i+1)
        def objective(trial):
            m = trial.suggest_float('m', mS[0], mS[1],step=0.25)
            nI = trial.suggest_int('nI', nIS[0], nIS[1]) 
            n_layers = trial.suggest_int('n_layers', nLS[0], nLS[1])
            nR = [trial.suggest_int(f'nR_layer_{l}', nRS[0], nRS[1]) for l in range(n_layers)]
            nO = trial.suggest_int('nO', nOS[0], nOS[1]) 
            N = trial.suggest_int('N', NS[0], NS[1])
            τ = trial.suggest_int('τ', τS[0], τS[1])
            mode = trial.suggest_int('mode', mdS[0], mdS[1])  
            act = trial.suggest_int('act', actS[0], actS[1])  

            if mode == 'past' or mode == 0:
                mO = trial.suggest_int('mO', 0, 0) 
                if nO > nI: raise TrialPruned()

            elif mode == 'ahead' or mode == 1:
                mO = trial.suggest_int('mO', mOS[0], mOS[1]) 
                if   nI > 20 or nO > 20: raise TrialPruned()
                elif mO > nI: raise TrialPruned()
                
            X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
            
            teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N],mode=mode,act=act,
                           tau=τ,rho=0.0,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
            
            for j,_ in enumerate(X[:]):
                teda.run(X[j])
                teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1)
                teda.Adapt(Y[j],Z[j])
                #teda.WAPE_HI2(Y[j],Z[j])

                if np.isinf(teda.hiP[-1]) or np.isnan(teda.hiP[-1]): raise TrialPruned()
                if not OptPrune: continue
        
                if j >=60 and j%5==0:
                    if teda.wape_RUL > 0.6: raise TrialPruned()
                    if teda.wape_HI  > 0.3: raise TrialPruned()

                if SingleObj and j%35==0:
                    trial.report(teda.wape_RUL+teda.wape_HI, j)
                    if trial.should_prune():
                        raise optuna.TrialPruned()

            if SingleObj: return teda.wape_RUL + teda.wape_HI
            else: return teda.wape_RUL,teda.wape_HI

        pruner=optuna.pruners.HyperbandPruner()
        if SingleObj: study = optuna.create_study(direction='minimize',pruner=pruner,sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        else: study = optuna.create_study(directions=['minimize','minimize'],sampler=SelSampler(mode=OptSampler,seed=OptSeed))
        study.optimize(objective, n_trials=n_trials, timeout=timeout, callbacks=[EarlyStoppingCallback(patience=patience)])

        vec = []
        if SingleObj:
            trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
            for trial in trials:
                p = trial.params
                nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
                row = [trial.values[0],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N'],p['τ'],p['mode'],p['act']]   
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
            df2 = df2.sort_values(by=df2.columns[0], ascending=False)[-5:]

        elif not SingleObj: 
            for trial in study.best_trials:
                p = trial.params
                nR_list = [p[f'nR_layer_{l}'] for l in range(p['n_layers'])]
                row = [trial.values[0],trial.values[1],p['m'],p['nI'],str(nR_list),p['nO'],p['mO'],p['N'],p['τ'],p['mode'],p['act']]
                vec.append(row)
            df2 = pd.DataFrame(vec,columns=names)
        
    if os.path.isfile(out_path) and out_path.startswith(study_dir):
        df1 = pd.read_csv(out_path)
        df_stdy = pd.concat([df1, df2], ignore_index=True)
    
    else: df_stdy = df2
    
    df_stdy.to_csv(out_path, index=False)
    opt_path = os.path.join(study_dir,f'opt_{len(os.listdir(study_dir))-1}.csv')
    
    if df2.shape[0] > 0: df2.to_csv(opt_path, index=False)

    return [df_stdy, df2 ]

In [ ]:
dfs = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler=None,OptPrune=True,
                         n_study=1,timeout=60,n_trials=0.5e3,patience=None,
                         mS=[2.0,4.5],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [13]:
for i in range(1,6):
    dfs = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler=None,OptPrune=True,
                            n_study=6,timeout=1680,n_trials=2e4,patience=2e3,
                            mS=[2.0,4.5],nLS=[i,i],nRS=[1,70],mdS=[1,1],actS=[0,1])

[I 2026-08-13 16:34:51,822] A new study created in memory with name: no-name-81c47cfe-0c75-455c-9edf-4fb5d1476057
[I 2026-08-13 16:34:51,823] Trial 0 pruned. 
[I 2026-08-13 16:34:51,825] Trial 1 pruned. 
[I 2026-08-13 16:34:51,828] Trial 2 pruned. 
[I 2026-08-13 16:34:51,829] Trial 3 pruned. 
[I 2026-08-13 16:34:51,832] Trial 4 pruned. 
[I 2026-08-13 16:34:51,834] Trial 5 pruned. 
[I 2026-08-13 16:34:51,836] Trial 6 pruned. 
[I 2026-08-13 16:34:51,838] Trial 7 pruned. 
[I 2026-08-13 16:34:51,853] Trial 8 pruned. 
[I 2026-08-13 16:34:51,856] Trial 9 pruned. 
[I 2026-08-13 16:34:51,857] Trial 10 pruned. 
[I 2026-08-13 16:34:51,859] Trial 11 pruned. 
[I 2026-08-13 16:34:51,861] Trial 12 pruned. 
[I 2026-08-13 16:34:51,862] Trial 13 pruned. 
[I 2026-08-13 16:34:51,864] Trial 14 pruned. 
[I 2026-08-13 16:34:51,867] Trial 15 pruned. 
[I 2026-08-13 16:34:51,868] Trial 16 pruned. 
[I 2026-08-13 16:34:51,870] Trial 17 pruned. 
[I 2026-08-13 16:34:51,872] Trial 18 pruned. 
[I 2026-08-13 16:34:51

iteration: 1


[I 2026-08-13 16:34:52,165] Trial 37 pruned. 
[I 2026-08-13 16:34:52,167] Trial 38 pruned. 
[I 2026-08-13 16:34:52,169] Trial 39 pruned. 
[I 2026-08-13 16:34:52,171] Trial 40 pruned. 
[I 2026-08-13 16:34:52,172] Trial 41 pruned. 
[I 2026-08-13 16:34:52,175] Trial 42 pruned. 
[I 2026-08-13 16:34:52,177] Trial 43 pruned. 
[I 2026-08-13 16:34:52,179] Trial 44 pruned. 
[I 2026-08-13 16:34:52,181] Trial 45 pruned. 
[I 2026-08-13 16:34:52,182] Trial 46 pruned. 
[I 2026-08-13 16:34:52,184] Trial 47 pruned. 
[I 2026-08-13 16:34:52,204] Trial 48 pruned. 
[I 2026-08-13 16:34:52,206] Trial 49 pruned. 
[I 2026-08-13 16:34:52,207] Trial 50 pruned. 
[I 2026-08-13 16:34:52,209] Trial 51 pruned. 
[I 2026-08-13 16:34:52,497] Trial 52 pruned. 
[I 2026-08-13 16:34:52,499] Trial 53 pruned. 
[I 2026-08-13 16:34:52,501] Trial 54 pruned. 
[I 2026-08-13 16:34:52,502] Trial 55 pruned. 
[I 2026-08-13 16:34:52,504] Trial 56 pruned. 
[I 2026-08-13 16:34:52,506] Trial 57 pruned. 
[I 2026-08-13 16:34:52,508] Trial 

iteration: 2


[I 2026-08-13 16:39:16,520] Trial 20 pruned. 
[I 2026-08-13 16:39:16,523] Trial 21 pruned. 
[I 2026-08-13 16:39:16,524] Trial 22 pruned. 
[I 2026-08-13 16:39:16,525] Trial 23 pruned. 
[I 2026-08-13 16:39:16,528] Trial 24 pruned. 
[I 2026-08-13 16:39:16,529] Trial 25 pruned. 
[I 2026-08-13 16:39:16,531] Trial 26 pruned. 
[I 2026-08-13 16:39:16,533] Trial 27 pruned. 
[I 2026-08-13 16:39:16,535] Trial 28 pruned. 
[I 2026-08-13 16:39:16,536] Trial 29 pruned. 
[I 2026-08-13 16:39:16,539] Trial 30 pruned. 
[I 2026-08-13 16:39:16,542] Trial 31 pruned. 
[I 2026-08-13 16:39:16,545] Trial 32 pruned. 
[I 2026-08-13 16:39:16,547] Trial 33 pruned. 
[I 2026-08-13 16:39:16,578] Trial 34 pruned. 
[I 2026-08-13 16:39:16,580] Trial 35 pruned. 
[I 2026-08-13 16:39:16,583] Trial 36 pruned. 
[I 2026-08-13 16:39:16,586] Trial 37 pruned. 
[I 2026-08-13 16:39:16,589] Trial 38 pruned. 
[I 2026-08-13 16:39:16,591] Trial 39 pruned. 
[I 2026-08-13 16:39:16,593] Trial 40 pruned. 
[I 2026-08-13 16:39:16,595] Trial 

KeyboardInterrupt: 

In [9]:
df = dfs[1]
#df = df[(df.iloc[:,0] <= 0.07)]
params_list = df.values[:,-9:]
df

,MAPE_RUL,MAPE_HI,m,nI,nR,nO,mO,N,TAU,past/ahead,activation


In [ ]:
tedas = []
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1],mode=mode,act=act,
                tau=τ,rho=0.03,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])
    

In [ ]:
PlotDSI_3D_PLT(tedas[0])